In [ ]:
from pathlib import Path

import hvplot.polars  # noqa: F401
import polars as pl

pl.Config.set_tbl_hide_column_data_types(True)

In [ ]:
def get_cap_mapping(data_dir, league_id):
    """
    Scans universe_info.csv files for a specific league to create a mapping 
    of Year to Salary Cap.
    
    Args:
        data_dir (str or Path): The root directory containing league data.
        league_id (str): The league identifier (e.g., 'DRAFT003').
    """
    # Ensure data_dir is a Path object for robust path joining
    data_path = Path(data_dir)
    
    # Construct the glob pattern: {data_dir}/{league_id}/*/universe_info.csv
    # scan_csv expects a string pattern for globbing
    glob_pattern = str(data_path / league_id / "*" / "universe_info.csv")
    
    # Create a dynamic regex pattern to extract the year following the league_id
    # Matches 'league_id/year/' and captures the digits (the year)
    year_regex = rf"{league_id}/(\d+)/"

    return (
        pl.scan_csv(glob_pattern, include_file_paths="filepath")
        .filter(pl.col("Information") == "Salary Cap (in tens of thousands)")
        .select([
            # Extract year from the file path and cast to integer
            pl.col("filepath").str.extract(year_regex, 1).cast(pl.Int32).alias("Year"),
            # Salary Cap = Value * 10,000 (as documented in universe_info.csv)
            (pl.col("Value/Round/Position").cast(pl.Int32) * 10_000).alias("Salary_Cap")
        ])
    ).collect()


In [ ]:
data_dir =  "../../data-generation/data/raw"
league_id = "DRAFT003"

In [ ]:
cap_map = get_cap_mapping(data_dir, league_id)
cap_map

In [ ]:
# Scan all annual player records
data_path = Path(data_dir)
glob_pattern_player_record = str(data_path / league_id / "*" / "player_record.csv")
glob_pattern_universe_info = str(data_path / league_id / "*" / "universe_info.csv")
year_regex = rf"{league_id}/(\d+)/"
annual_earnings = (
    pl.scan_csv(glob_pattern_player_record, include_file_paths="filepath")
    .with_columns(
        pl.col("filepath").str.extract(year_regex, 1).cast(pl.Int32).alias("Year")
    )
    .join(cap_map.lazy(), on="Year")
    .with_columns(
        ((pl.col("Salary_Year_1") + pl.col("Bonus_Year_1")) * 10_000 / pl.col("Salary_Cap"))
        .alias("Annual_Cap_Share")
    )
    .group_by("Player_ID")
    .agg(pl.col("Annual_Cap_Share").sum().alias("Total_Career_Cap_Share"))
)

In [ ]:
annual_earnings_df = annual_earnings.collect()

In [ ]:
annual_earnings_df.sort(pl.col("Total_Career_Cap_Share"), descending=True)

In [ ]:
# Check how many rows (years) we have for our top earner
top_player_id = annual_earnings_df.sort("Total_Career_Cap_Share", descending=True)[0, "Player_ID"]

verification = (
    pl.scan_csv(glob_pattern_player_record)
    .filter(pl.col("Player_ID") == top_player_id)
    .select(pl.len().alias("Years_Found"))
    .collect()
)

print(f"Top Player Years Found: {verification['Years_Found'][0]}")

In [ ]:
# Ensure we use the correct row/column indexing for the top player ID
top_player_id = annual_earnings_df.sort("Total_Career_Cap_Share", descending=True)[0, "Player_ID"]

debug_df = (
    pl.scan_csv(
        glob_pattern_player_record,
        include_file_paths="filepath" # Enables the filepath column
    )
    .filter(pl.col("Player_ID") == top_player_id)
    .with_columns(
        # Extract the Year from the path so we can join with the cap_map
        pl.col("filepath").str.extract(year_regex, 1).cast(pl.Int32).alias("Year")
    )
    .join(cap_map.lazy(), on="Year") 
    .select([
        "Year",
        "Player_ID",
        "Salary_Year_1",
        "Bonus_Year_1",
        "Salary_Cap", # Total dollar amount (Value * 10,000)
        # Normalization: (Salary*10,000 + Bonus*1,000) / (Cap_Value * 10,000)
        # Which simplifies to (Salary + Bonus) / (Cap_Value * 10)
        (((pl.col("Salary_Year_1") + pl.col("Bonus_Year_1")) * 10_000) / pl.col("Salary_Cap")).alias("Annual_Pct")
    ])
    .sort("Year")
    .collect()
)

with pl.Config(tbl_rows=-1):
    display(debug_df)

In [ ]:
# Find the single highest annual cap hits in league history
superstar_check = (
    pl.scan_csv(glob_pattern_player_record, include_file_paths="filepath")
    .with_columns(
        pl.col("filepath").str.extract(year_regex, 1).cast(pl.Int32).alias("Year")
    )
    .join(cap_map.lazy(), on="Year")
    .with_columns(
        (((pl.col("Salary_Year_1") + pl.col("Bonus_Year_1")) * 10_000) / pl.col("Salary_Cap")).alias("Annual_Pct")
    )
    .select(["Year", "Player_ID", "Salary_Year_1", "Bonus_Year_1", "Annual_Pct"])
    .sort("Annual_Pct", descending=True)
    .head(10)
    .collect()
)

with pl.Config(tbl_rows=-1):
    display(superstar_check)

In [ ]:
# Check raw values for a year from your previous output (e.g., 2115)
raw_cap = pl.read_csv(glob_pattern_universe_info)
display(raw_cap.filter(pl.col("Information").str.contains("Salary Cap")))

raw_player = pl.read_csv(glob_pattern_player_record).filter(pl.col("Player_ID") == 7362)
display(raw_player.select(["Player_ID", "Salary_Year_1", "Bonus_Year_1"]))

In [ ]:
annual_earnings_df.filter(pl.col("Total_Career_Cap_Share")>0).hvplot.hist(y="Total_Career_Cap_Share", bins="auto")

In [ ]:
annual_earnings_df.filter(pl.col("Total_Career_Cap_Share")>=.5).shape[0]